# Lab 18 · Cechy i drzewa

Przed wykonaniem tego laboratorium przeczytaj wykłady [D15.1 „Zapisz kalendarz i kategorie jako cechy”](../lectures/d15-calendar.html), [D15.2 „Prześledź podział drzewa”](../lectures/d15-tree.html) i [D15.3 „Połącz drzewa w las i boosting”](../lectures/d15-ensembles.html).

Pamiętaj, że do tego labu potrzebne jest środowisko z labu 9: projekt practice z bibliotekami sekcji C. Uruchom Jupyter tak samo jak tam.

Pod opisem każdego zadania znajduje się komórka kodu do uzupełnienia, a pod nią komórka sprawdzenia. Jeżeli zadanie nie jest jeszcze rozwiązane, sprawdzenie zatrzyma się i wyświetli czerwony komunikat. To normalne: komunikat mówi, co trzeba poprawić.

Najpierw przewidź wynik, potem uruchom kod. Podpowiedzi otwieraj pojedynczo.

In [ ]:
from pathlib import Path
import sys
assert sys.version_info[:2] == (3, 12), "Wybierz kernel Pythona 3.12 z głównego README."
lab_dir = Path.cwd()
assert (lab_dir / "lab-18-cechy-i-drzewa.ipynb").is_file(), "Uruchom Jupyter z folderu labs."
print("Folder:", lab_dir)

In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError as error:
    raise AssertionError("Brakuje bibliotek sekcji C. Dodaj je w practice i uruchom Jupyter przez --project ../practice, zgodnie z głównym README.") from error
print("pandas", pd.__version__, "| numpy", np.__version__)

## Dane projektu z kalendarzem

Gotowa komórka odtwarza `train` i `valid` z C12 i dodaje kolumnę `weekday`, czyli numer dnia tygodnia znany z wyprzedzeniem dla każdej daty. Zero to poniedziałek.

In [ ]:
sales = pd.read_csv(lab_dir / "data" / "project-development.csv", parse_dates=["day"])
assert sales.shape == (49, 2), "Oczekiwano 49 dni i dwóch kolumn."
cutoff = pd.Timestamp("2024-02-12")
with_lag = sales.assign(lag1=sales["sale"].shift(1))
rows = with_lag[with_lag["lag1"].notna()]
train = rows[rows["day"] < cutoff]
valid = rows[rows["day"] >= cutoff]
assert len(train) == 41 and len(valid) == 7
print("Trening:", train["day"].min().date(), "–", train["day"].max().date(), "| walidacja:", valid["day"].min().date(), "–", valid["day"].max().date())
train = train.assign(weekday=train["day"].dt.dayofweek)
valid = valid.assign(weekday=valid["day"].dt.dayofweek)
print(train[["day", "weekday", "lag1", "sale"]].head(3))

## D15.1 · Dzień tygodnia jako kategoria

Po wykładzie D15.1 w `prepare` zbuduj `ColumnTransformer` z jednym etapem o nazwie `"weekday"`: `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` dla kolumny `weekday`, z `remainder="passthrough"` dla `lag1`. Nazwa etapu jest potrzebna sprawdzeniu. Dopasuj go wyłącznie na `train[["weekday", "lag1"]]`. W `train_matrix` zapisz wynik `fit_transform` na treningu, a w `valid_matrix` wynik `transform` na walidacji. Przewidź kształty obu macierzy.

In [ ]:
prepare = None
train_matrix = None
valid_matrix = None

In [ ]:
from sklearn.compose import ColumnTransformer
assert isinstance(prepare, ColumnTransformer), "D15.1: prepare ma być ColumnTransformer z etapem OneHotEncoder."
assert "weekday" in prepare.named_transformers_, "D15.1: nazwij etap kodera weekday, tak jak w poleceniu."
categories = list(prepare.named_transformers_["weekday"].categories_[0])
assert categories == [0, 1, 2, 3, 4, 5, 6], "D15.1: koder poznaje siedem dni tygodnia z treningu."
assert isinstance(train_matrix, np.ndarray) and train_matrix.shape == (41, 8), "D15.1: 41 wierszy treningu, 7 kolumn wskaźnikowych plus lag1."
assert isinstance(valid_matrix, np.ndarray) and valid_matrix.shape == (7, 8), "D15.1: walidację przekształcasz tym samym koderem przez transform, bez ponownego fit."
assert np.allclose(train_matrix[:, :7].sum(axis=1), 1) and np.allclose(train_matrix[:, 7], train["lag1"]), "D15.1: każdy wiersz ma jedną jedynkę wśród dni tygodnia, a lag1 przechodzi bez zmian."
print("D15.1: kalendarz jest zakodowany na treningu i zastosowany do walidacji.")

<details><summary>Podpowiedź 1</summary>

Przykład wykładu D15.1 pokazuje krotkę (nazwa, koder, lista kolumn) i remainder="passthrough".

</details>

<details><summary>Podpowiedź 2</summary>

fit_transform na treningu, transform na walidacji. Nazwa etapu "weekday" pozwala sprawdzeniu odczytać kategorie.

</details>

## D15.1 · Projekt z kalendarzem

W `calendar_model` zbuduj `make_pipeline` z nowym `ColumnTransformer` jak wyżej i `Ridge(alpha=1.0)`, dopasowany na `train[["weekday", "lag1"]]` i `train["sale"]`. W `calendar_mae` zapisz MAE na walidacji. Porównaj z wynikami z pliku results/c12-project.json; zapisz także wynik, jeśli kalendarz nie pomógł.

In [ ]:
calendar_model = None
calendar_mae = None

In [ ]:
import math
from sklearn.pipeline import Pipeline
assert isinstance(calendar_model, Pipeline), "D15.1: calendar_model to pipeline z koderem i Ridge."
assert calendar_mae is not None and math.isclose(float(calendar_mae), 2.713614453041519, rel_tol=1e-6), "D15.1: dopasuj pipeline tylko na treningu i policz MAE na siedmiu dniach walidacji."
print("D15.1: MAE z kalendarzem na walidacji:", round(float(calendar_mae), 4), "[szt.]")

<details><summary>Podpowiedź 1</summary>

make_pipeline(ColumnTransformer(...), Ridge(alpha=1.0)); fit na dwóch kolumnach cech.

</details>

<details><summary>Podpowiedź 2</summary>

MAE licz jak w C12: wartość bezwzględna różnicy valid["sale"] i predict, potem mean.

</details>

## D15.2 · Podział drzewa na kartce

Po wykładzie D15.2 sześć obserwacji w `tx` i `ty` dzielimy regułą `x <= 3`. W `left_mean` i `right_mean` zapisz średnie celu w obu grupach. W `sse_before` zapisz sumę kwadratów odchyleń od wspólnej średniej przed podziałem, a w `sse_after` sumę takich sum w obu grupach. W `manual_predictions` zapisz listę prognoz drzewa dla x równego 4 i 100.

In [ ]:
tx = np.array([1.0, 2.0, 3.0, 6.0, 7.0, 9.0])
ty = np.array([2.0, 3.0, 4.0, 8.0, 9.0, 10.0])
left_mean = None
right_mean = None
sse_before = None
sse_after = None
manual_predictions = None

In [ ]:
import math
assert left_mean is not None and math.isclose(float(left_mean), 3.0) and right_mean is not None and math.isclose(float(right_mean), 9.0), "D15.2: lewa grupa to x <= 3 ze średnią 3, prawa ma średnią 9."
assert sse_before is not None and math.isclose(float(sse_before), 58.0), "D15.2: wspólna średnia to 6; suma kwadratów odchyleń wynosi 58."
assert sse_after is not None and math.isclose(float(sse_after), 4.0), "D15.2: po podziale każda grupa ma sumę kwadratów 2; razem 4."
assert isinstance(manual_predictions, list) and np.allclose(manual_predictions, [9.0, 9.0]), "D15.2: 4 i 100 trafiają do prawego liścia; drzewo nie przedłuża trendu."
print("D15.2: podział obniżył sumę kwadratów z 58 do 4; prognoza to średnia liścia.")

<details><summary>Podpowiedź 1</summary>

Maska tx <= 3 wybiera trzy pierwsze obserwacje; średnia celu to 3.

</details>

<details><summary>Podpowiedź 2</summary>

Suma kwadratów to sum((ty - mean) ** 2). Prognoza dla x = 100 to średnia liścia, do którego trafia.

</details>

## D15.2 · To samo drzewo w scikit-learn

W `tree` dopasuj `DecisionTreeRegressor(max_depth=1, random_state=0)` na `tx.reshape(-1, 1)` i `ty`. W `tree_predictions` zapisz prognozy dla x równego 4 i 100 jako listę. Biblioteka stawia próg w połowie odstępu między obserwacjami 3 i 6, czyli w 4.5, a nie dokładnie w 3 jak ręczna reguła. Przewidź, do którego liścia trafi teraz x równe 4, zanim uruchomisz kod.

In [ ]:
tree = None
tree_predictions = None

In [ ]:
from sklearn.tree import DecisionTreeRegressor
assert isinstance(tree, DecisionTreeRegressor) and tree.get_depth() == 1, "D15.2: dopasuj DecisionTreeRegressor o głębokości 1."
assert isinstance(tree_predictions, list) and np.allclose(tree_predictions, [3.0, 9.0]), "D15.2: predict potrzebuje tablicy o kształcie (2, 1); próg 4.5 kieruje x = 4 do lewego liścia, a x = 100 do prawego."
print("D15.2: próg biblioteki 4.5 zmienia prognozę dla x = 4; prognoza nadal jest średnią liścia.")

<details><summary>Podpowiedź 1</summary>

Przykład wykładu D15.2 pokazuje fit na liście list i predict na nowych wartościach.

</details>

<details><summary>Podpowiedź 2</summary>

Wynik predict jest tablicą; zamień go na listę metodą tolist. Wartość 4 jest mniejsza od progu 4.5.

</details>

## D15.3 · Drzewo, las i boosting na tych samych wierszach

Po wykładzie D15.3 porównaj na cechach `["weekday", "lag1"]` trzy modele dopasowane na `train`: `DecisionTreeRegressor(max_depth=3, random_state=0)`, `RandomForestRegressor(n_estimators=50, random_state=0)` i `GradientBoostingRegressor(n_estimators=50, learning_rate=0.1, max_depth=2, random_state=0)`. W `ensemble_mae` zapisz słownik z kluczami `tree`, `forest`, `boosting` i MAE na walidacji. W `best_tree_model` zapisz klucz najlepszego wyniku. Ustawienia zostają takie, jak podano; zmieniaj jedną rzecz naraz dopiero po zapisaniu tego wyniku.

In [ ]:
ensemble_mae = None
best_tree_model = None

In [ ]:
import math
expected = {'tree': 3.488972431077692, 'forest': 3.276142857142859, 'boosting': 2.9664543280737843}
assert isinstance(ensemble_mae, dict) and set(ensemble_mae) == set(expected), "D15.3: słownik z kluczami tree, forest i boosting."
for key, wanted in expected.items():
    assert ensemble_mae[key] is not None and math.isclose(float(ensemble_mae[key]), wanted, rel_tol=1e-3), f"D15.3: {key}: użyj podanych ustawień i random_state=0, fit na train, MAE na valid."
assert best_tree_model == min(expected, key=expected.get), "D15.3: best_tree_model to klucz z najniższym MAE walidacji."
print("D15.3: MAE na walidacji:", {key: round(float(value), 4) for key, value in ensemble_mae.items()}, "[szt.]")

<details><summary>Podpowiedź 1</summary>

Słownik nazwa -> model, pętla po nim z fit i predict; MAE jak wcześniej.

</details>

<details><summary>Podpowiedź 2</summary>

min(ensemble_mae, key=ensemble_mae.get) zwraca klucz najmniejszej wartości.

</details>

## D15.3 · Jeden krok boostingu na kartce

Bieżąca prognoza dwóch dni to `current`, a nowe drzewo zwraca korektę `correction`. W `updated` zapisz listę dwóch prognoz po dodaniu korekty pomnożonej przez `rate`. To nie jest średnia dwóch modeli.

In [ ]:
current = [10.0, 4.0]
correction = [-2.0, 6.0]
rate = 0.25
updated = None

In [ ]:
assert isinstance(updated, list) and np.allclose(updated, [9.5, 5.5]), "D15.3: nowa prognoza to current plus rate razy correction, dzień po dniu."
print("D15.3: boosting dodaje przeskalowaną korektę do bieżącej prognozy.")

<details><summary>Podpowiedź 1</summary>

Pierwszy dzień: 10 + 0.25 * (-2).

</details>

<details><summary>Podpowiedź 2</summary>

zip(current, correction) i comprehension dają listę dwóch liczb.

</details>

## Sprawdź rezultat

Sprawdzenia przechodzą po restarcie kernela. Do przemyślenia: dlaczego koder dopasowany osobno na walidacji mógłby zmienić znaczenie kolumn? Czy drzewo przewidziałoby sprzedaż wyższą niż najwyższa w treningu? Który z trzech modeli drzewiastych pokonał baseline z C12, a który nie? Nie musisz spisywać odpowiedzi.